# Flight Line Operations

Flight lines are the fundamental building block of airborne remote sensing campaigns.
A `FlightLine` represents a straight-path segment that the aircraft flies while
collecting data. This notebook provides comprehensive coverage of HyPlan's
`FlightLine` class and related utilities.

We cover:

1. Creating flight lines (start-point and center-point constructors)
2. Inspecting flight line properties (length, azimuth, waypoints)
3. Generating interpolated tracks
4. Clipping flight lines to study area polygons
5. Splitting long flight lines into segments with gaps
6. Offsetting flight lines (along-track and across-track)
7. Rotating flight lines around the midpoint
8. Reversing flight direction
9. North/East offsets
10. Converting to GeoDataFrame, GeoJSON, and dictionary formats

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import geopandas as gpd

from hyplan.flight_line import FlightLine, to_gdf
from hyplan.units import ureg

## 1. Creating Flight Lines

HyPlan provides two class methods for creating flight lines:

- **`start_length_azimuth`** — Define by start point, length, and heading
- **`center_length_azimuth`** — Define by center point, length, and heading (line extends equally in both directions)

In [ ]:
# Method 1: Start point + length + azimuth
# A 50 km line heading northeast from downtown Los Angeles
fl_start = FlightLine.start_length_azimuth(
    lat1=34.05,
    lon1=-118.25,
    length=ureg.Quantity(50, "km"),
    az=45.0,
    altitude_msl=ureg.Quantity(20000, "feet"),
    site_name="LA Northeast",
    investigator="Dr. Smith",
)

# Method 2: Center point + length + azimuth
# An 80 km east-west line centered on a point in the San Fernando Valley
fl_center = FlightLine.center_length_azimuth(
    lat=34.15,
    lon=-118.55,
    length=ureg.Quantity(80, "km"),
    az=80.0,
    altitude_msl=ureg.Quantity(20000, "feet"),
    site_name="SFV East-West",
    investigator="Dr. Jones",
)

fig, ax = plt.subplots(figsize=(10, 6))
for fl, color, marker in [(fl_start, "blue", "o"), (fl_center, "green", "s")]:
    x, y = zip(*fl.geometry.coords)
    ax.plot(x, y, marker=marker, color=color, label=fl.site_name)
    ax.annotate("Start", (x[0], y[0]), fontsize=8, color=color)
    ax.annotate("End", (x[-1], y[-1]), fontsize=8, color=color)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Flight Line Creation Methods")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Inspecting Flight Line Properties

Every `FlightLine` exposes geodesic properties computed via the Vincenty formula:
latitude/longitude of endpoints, geodesic length, forward and reverse azimuths,
and waypoint representations for integration with the Dubins path planner.

In [ ]:
fl = fl_start

print(f"Site name:    {fl.site_name}")
print(f"Investigator: {fl.investigator}")
print(f"Altitude MSL: {fl.altitude_msl}")
print()
print(f"Start point:  ({fl.lat1:.4f}, {fl.lon1:.4f})")
print(f"End point:    ({fl.lat2:.4f}, {fl.lon2:.4f})")
print(f"Length:        {fl.length:.1f}  ({fl.length.to('km'):.2f})")
print(f"Forward az:    {fl.az12:.2f}")
print(f"Reverse az:    {fl.az21:.2f}")
print()

# Waypoint representations (used by DubinsPath and flight_plan)
wp1 = fl.waypoint1
wp2 = fl.waypoint2
print(f"Waypoint 1: lat={wp1.latitude:.4f}, lon={wp1.longitude:.4f}, heading={wp1.heading:.1f}")
print(f"Waypoint 2: lat={wp2.latitude:.4f}, lon={wp2.longitude:.4f}, heading={wp2.heading:.1f}")

## 3. Interpolated Track

The `track()` method generates a densely-sampled `LineString` along the geodesic,
useful for visualization and swath calculations. The `precision` parameter controls
the spacing between interpolated points (default 100 m).

In [ ]:
# Default track (100 m spacing)
track_default = fl_start.track()
print(f"Default track: {len(track_default.coords)} points (100 m spacing)")

# Coarser track (1 km spacing)
track_coarse = fl_start.track(precision=ureg.Quantity(1000, "meter"))
print(f"Coarse track:  {len(track_coarse.coords)} points (1 km spacing)")

# Fine track (50 m spacing)
track_fine = fl_start.track(precision=ureg.Quantity(50, "meter"))
print(f"Fine track:    {len(track_fine.coords)} points (50 m spacing)")

# Visualize the difference
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, track, title in [
    (axes[0], track_coarse, f"Coarse (1 km, {len(track_coarse.coords)} pts)"),
    (axes[1], track_fine, f"Fine (50 m, {len(track_fine.coords)} pts)"),
]:
    lons, lats = zip(*track.coords)
    ax.plot(lons, lats, ".-", markersize=3)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Clipping to a Study Area

`clip_to_polygon()` clips a flight line to a polygon boundary, returning a list of
`FlightLine` segments that fall within the polygon. This is essential when generating
flight boxes over irregular study areas — lines that extend beyond the boundary get
trimmed automatically.

In [ ]:
# Define an irregular study area polygon
study_area = Polygon([
    (-118.7, 34.0),
    (-118.6, 34.3),
    (-118.3, 34.2),
    (-118.4, 34.0),
    (-118.5, 34.1),
])

# Clip the center-defined flight line to the study area
clipped = fl_center.clip_to_polygon(study_area)

fig, ax = plt.subplots(figsize=(10, 7))

# Plot the study area
x, y = study_area.exterior.xy
ax.fill(x, y, alpha=0.1, color="red")
ax.plot(x, y, "--", color="red", label="Study area")

# Plot original flight line
ox, oy = zip(*fl_center.geometry.coords)
ax.plot(ox, oy, "b--", alpha=0.4, label="Original line")

# Plot clipped segments
if clipped:
    for i, seg in enumerate(clipped):
        sx, sy = zip(*seg.geometry.coords)
        ax.plot(sx, sy, "g-o", linewidth=2, label=f"Clipped segment {i + 1}")
    print(f"Clipping produced {len(clipped)} segment(s)")
    for i, seg in enumerate(clipped):
        print(f"  Segment {i + 1}: {seg.length.to('km'):.2f}")
else:
    print("Flight line does not intersect the study area")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Clipping a Flight Line to a Study Area Polygon")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Splitting into Segments

`split_by_length()` divides a long flight line into shorter segments, optionally
with gaps between them. This is useful for:
- Breaking long lines into manageable data-collection runs
- Inserting calibration gaps between segments
- Accommodating data system limitations on continuous recording time

In [ ]:
# Split into 10 km segments with no gap
segments_no_gap = fl_start.split_by_length(max_length=ureg.Quantity(10, "km"))
print(f"No gap: {len(segments_no_gap)} segments")

# Split into 10 km segments with 2 km gaps
segments_with_gap = fl_start.split_by_length(
    max_length=ureg.Quantity(10, "km"),
    gap_length=ureg.Quantity(2, "km"),
)
print(f"With 2 km gaps: {len(segments_with_gap)} segments")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, segs, title in [
    (axes[0], segments_no_gap, "No Gap"),
    (axes[1], segments_with_gap, "2 km Gaps"),
]:
    colors = plt.cm.tab10(np.linspace(0, 1, len(segs)))
    for i, seg in enumerate(segs):
        sx, sy = zip(*seg.geometry.coords)
        ax.plot(sx, sy, "o-", color=colors[i], label=f"Seg {i + 1} ({seg.length.to('km'):.1f})")
    ax.set_title(f"Split by Length — {title}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Along-Track and Across-Track Offsets

- **`offset_along()`** — Shortens or extends the line by moving its start and/or end
  points along the flight direction. Positive values move the point forward (in the
  direction of flight); negative values move it backward.
- **`offset_across()`** — Shifts the entire line perpendicular to its direction.
  Positive values offset to the right (looking along the flight direction).
- **`offset_north_east()`** — Shifts the entire line by fixed north and east distances.

In [ ]:
# Along-track offset: trim 10 km from start, 5 km from end
fl_trimmed = fl_start.offset_along(
    offset_start=ureg.Quantity(10, "km"),
    offset_end=ureg.Quantity(-5, "km"),
)
print(f"Original length: {fl_start.length.to('km'):.2f}")
print(f"Trimmed length:  {fl_trimmed.length.to('km'):.2f}")

# Across-track offset: shift 5 km to the right
fl_right = fl_start.offset_across(ureg.Quantity(5, "km"))

# Across-track offset: shift 5 km to the left
fl_left = fl_start.offset_across(ureg.Quantity(-5, "km"))

# North-East offset: shift 10 km north and 5 km east
fl_ne = fl_start.offset_north_east(
    offset_north=ureg.Quantity(10, "km"),
    offset_east=ureg.Quantity(5, "km"),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Along-track
ax = axes[0]
for line, label, style in [
    (fl_start, "Original", "b-o"),
    (fl_trimmed, "Trimmed (+10 km start, -5 km end)", "r-s"),
]:
    x, y = zip(*line.geometry.coords)
    ax.plot(x, y, style, label=label)
ax.set_title("Along-Track Offset")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Across-track and North-East
ax = axes[1]
for line, label, style in [
    (fl_start, "Original", "b-o"),
    (fl_right, "Right +5 km", "g-s"),
    (fl_left, "Left -5 km", "r-^"),
    (fl_ne, "North +10 km, East +5 km", "m-d"),
]:
    x, y = zip(*line.geometry.coords)
    ax.plot(x, y, style, label=label, markersize=5)
ax.set_title("Across-Track and North-East Offsets")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Rotation

`rotate_around_midpoint()` rotates the flight line around its geographic midpoint
by a specified angle in degrees (clockwise positive). This is useful for exploring
different flight headings over the same study site.

In [ ]:
# Rotate in 30-degree increments
angles = [0, 30, 60, 90, 120, 150]

fig, ax = plt.subplots(figsize=(9, 8))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(angles)))

for angle, color in zip(angles, colors):
    rotated = fl_start.rotate_around_midpoint(angle=angle)
    x, y = zip(*rotated.geometry.coords)
    ax.plot(x, y, "-o", color=color, markersize=4,
            label=f"+{angle} deg (az={rotated.az12:.0f} deg)")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Rotation Around Midpoint")
ax.legend()
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Reversing Direction

`reverse()` swaps the start and end points, which flips the flight direction.
This is useful when optimizing flight line ordering to minimize ferry time
between consecutive lines.

In [ ]:
fl_reversed = fl_start.reverse()

print(f"Original:  start=({fl_start.lat1:.4f}, {fl_start.lon1:.4f}), "
      f"end=({fl_start.lat2:.4f}, {fl_start.lon2:.4f}), az={fl_start.az12:.1f}")
print(f"Reversed:  start=({fl_reversed.lat1:.4f}, {fl_reversed.lon1:.4f}), "
      f"end=({fl_reversed.lat2:.4f}, {fl_reversed.lon2:.4f}), az={fl_reversed.az12:.1f}")

fig, ax = plt.subplots(figsize=(10, 5))

for fl_plot, label, color in [
    (fl_start, "Original", "blue"),
    (fl_reversed, "Reversed", "orange"),
]:
    coords = list(fl_plot.geometry.coords)
    x, y = zip(*coords)
    ax.plot(x, y, color=color, linewidth=2, label=label)
    # Arrow at midpoint to show direction
    mid = len(coords) // 2
    ax.annotate("", xy=coords[mid], xytext=coords[mid - 1],
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2))
    ax.plot(x[0], y[0], "o", color=color, markersize=8)  # Start marker

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Reversing Flight Direction (circles = start points)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Serialization and Export

HyPlan provides several ways to serialize and export flight lines:

- **`to_dict()`** — Python dictionary with all attributes
- **`to_geojson()`** — GeoJSON Feature dictionary
- **`to_gdf()`** — Convert a list of flight lines to a GeoDataFrame for analysis or file export

In [ ]:
# Dictionary representation
d = fl_start.to_dict()
for key, val in d.items():
    print(f"  {key}: {val}")

In [ ]:
# GeoJSON Feature
import json

geojson = fl_start.to_geojson()
print(json.dumps(geojson, indent=2)[:500], "...")

In [ ]:
# Convert multiple flight lines to a GeoDataFrame
flight_lines = [fl_start, fl_center, fl_trimmed, fl_right]
gdf = to_gdf(flight_lines)
print(f"GeoDataFrame with {len(gdf)} flight lines:")
gdf

In [ ]:
# Export to various formats via GeoPandas
# gdf.to_file("flight_lines.geojson", driver="GeoJSON")
# gdf.to_file("flight_lines.shp")
# gdf.to_file("flight_lines.gpkg", driver="GPKG")

# Quick plot of the GeoDataFrame
gdf.plot(figsize=(10, 6), column="site_name", legend=True, linewidth=2)
plt.title("Flight Lines GeoDataFrame")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Feature | Method | Purpose |
|---------|--------|---------|
| Create from start point | `FlightLine.start_length_azimuth()` | Define line by start, length, heading |
| Create from center point | `FlightLine.center_length_azimuth()` | Define line centered on a point |
| Endpoints & geometry | `.lat1`, `.lon1`, `.lat2`, `.lon2`, `.geometry` | Access geographic coordinates |
| Geodesic length | `.length` | Vincenty-computed line length |
| Azimuths | `.az12`, `.az21` | Forward and reverse headings |
| Waypoints | `.waypoint1`, `.waypoint2` | Integration with Dubins path planner |
| Interpolated track | `.track(precision)` | Dense point sampling along geodesic |
| Clip to polygon | `.clip_to_polygon(polygon)` | Trim to study area boundary |
| Split into segments | `.split_by_length(max_length, gap_length)` | Break into shorter runs |
| Along-track offset | `.offset_along(start, end)` | Trim or extend endpoints |
| Across-track offset | `.offset_across(distance)` | Shift perpendicular to flight direction |
| North/East offset | `.offset_north_east(north, east)` | Shift by fixed cardinal distances |
| Rotate | `.rotate_around_midpoint(angle)` | Explore different headings |
| Reverse | `.reverse()` | Flip flight direction |
| Serialize | `.to_dict()`, `.to_geojson()` | Dictionary and GeoJSON export |
| Batch convert | `to_gdf(flight_lines)` | GeoDataFrame for analysis/export |